# Olist Logistics & SLA Diagnostic

* **Environment:** SQLite (in-memory) + Pandas
* **Business Goal:** To diagnose fulfillment bottlenecks by separating Seller Handling Time from Carrier Transit Time, and ultimately export clean dimensions for Tableau visualization.

In [3]:
import sqlite3
import pandas as pd

# Create an in-memory SQLite database — simulates a lightweight data warehouse
conn = sqlite3.connect(':memory:')

# Define CSV files to load and their corresponding table names
csv_files = {
    'olist_orders_dataset':     'olist_orders_dataset.csv',
    'olist_customers_dataset':  'olist_customers_dataset.csv',
    'olist_order_items_dataset':'olist_order_items_dataset.csv',
    'olist_sellers_dataset':    'olist_sellers_dataset.csv',
}

# Load each CSV into a pandas DataFrame and persist it as a SQL table
for table_name, filename in csv_files.items():
    df = pd.read_csv(filename)
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f' {table_name} loaded  ({len(df):,} rows)')

print('\n All tables ready — SQL environment initialised.')

 olist_orders_dataset loaded  (99,441 rows)
 olist_customers_dataset loaded  (99,441 rows)
 olist_order_items_dataset loaded  (112,650 rows)
 olist_sellers_dataset loaded  (3,095 rows)

 All tables ready — SQL environment initialised.



## Query 1 — Overall Delivery KPI Summary


In [4]:
query_overall_kpi = """
SELECT
    -- Total number of successfully delivered orders
    COUNT(order_id) AS total_delivered,

    -- Orders where actual delivery exceeded the promised delivery date
    SUM(CASE
            WHEN order_delivered_customer_date > order_estimated_delivery_date
            THEN 1 ELSE 0
        END) AS total_delayed,

    -- Delay rate: percentage of orders that missed the SLA
    ROUND(
        SUM(CASE
                WHEN order_delivered_customer_date > order_estimated_delivery_date
                THEN 1 ELSE 0
            END) * 100.0 / COUNT(order_id) , 2) AS delay_rate_pct,

    -- On-time rate: the primary SLA compliance KPI
    ROUND(
        100 - SUM(CASE
                      WHEN order_delivered_customer_date > order_estimated_delivery_date
                      THEN 1 ELSE 0
                  END) * 100.0 / COUNT(order_id)
    , 2)                                                                     AS ontime_rate_pct,

    -- Average end-to-end delivery time from order placement to customer receipt
    ROUND(
        AVG(julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp))
    , 2)                                                                     AS avg_delivery_days

FROM olist_orders_dataset
WHERE
    order_status                    = 'delivered'
    AND order_delivered_customer_date  IS NOT NULL
    AND order_estimated_delivery_date  IS NOT NULL
    AND order_purchase_timestamp       IS NOT NULL;
"""

df_kpi = pd.read_sql(query_overall_kpi, conn)
display(df_kpi)

# Save SQL output for reuse in Pandas analysis — avoids redundant computation
kpi_ontime_rate  = df_kpi.loc[0, 'ontime_rate_pct']  
kpi_delay_rate   = df_kpi.loc[0, 'delay_rate_pct']    
kpi_avg_days     = df_kpi.loc[0, 'avg_delivery_days'] 

,total_delivered,total_delayed,delay_rate_pct,ontime_rate_pct,avg_delivery_days
0,96470,7826,8.11,91.89,12.56


---
## Query 2 — Top 5 States by Delay Rate

In [5]:
query_top5_states = """
SELECT
    c.customer_state                                                         AS state,
    COUNT(o.order_id)                                                        AS total_orders,

    SUM(CASE
            WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 1 ELSE 0
        END)                                                                 AS delayed_orders,

    -- Delay rate per state — the key metric for regional performance benchmarking
    ROUND(
        SUM(CASE
                WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
                THEN 1 ELSE 0
            END) * 100.0 / COUNT(o.order_id)
    , 2)                                                                     AS delay_rate_pct,

    -- Average days late (only for delayed orders) — measures severity, not just frequency
    ROUND(
        AVG(CASE
                WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
                THEN julianday(o.order_delivered_customer_date)
                         - julianday(o.order_estimated_delivery_date)
                ELSE NULL
            END)
    , 2)                                                                     AS avg_days_late

FROM olist_orders_dataset      o
JOIN olist_customers_dataset   c ON o.customer_id = c.customer_id
WHERE
    o.order_status                    = 'delivered'
    AND o.order_delivered_customer_date  IS NOT NULL
    AND o.order_estimated_delivery_date  IS NOT NULL
GROUP BY
    c.customer_state
HAVING
    -- Exclude low-volume states to ensure statistical reliability
    COUNT(o.order_id) > 50
ORDER BY
    delay_rate_pct DESC
LIMIT 5;
"""

df_top5 = pd.read_sql(query_top5_states, conn)
display(df_top5)

# Retain SQL-computed state rankings for validation against Pandas RCA results
sql_top5_states = df_top5['state'].tolist()  # ['AL', 'MA', 'SE', ...]

,state,total_orders,delayed_orders,delay_rate_pct,avg_days_late
0,AL,397,95,23.93,9.25
1,MA,717,141,19.67,9.97
2,PI,476,76,15.97,12.30
3,CE,1279,196,15.32,14.33
4,SE,335,51,15.22,16.89



## Query 3 — Monthly Delay Trend


In [6]:
query_monthly_trend = """
SELECT
    -- Truncate timestamp to year-month for monthly aggregation
    strftime('%Y-%m', order_purchase_timestamp)                              AS order_month,

    COUNT(order_id)                                                          AS total_orders,

    SUM(CASE
            WHEN order_delivered_customer_date > order_estimated_delivery_date
            THEN 1 ELSE 0
        END)                                                                 AS delayed_orders,

    -- Monthly delay rate — used to plot the trend line on the ops dashboard
    ROUND(
        SUM(CASE
                WHEN order_delivered_customer_date > order_estimated_delivery_date
                THEN 1 ELSE 0
            END) * 100.0 / COUNT(order_id)
    , 2)                                                                     AS delay_rate_pct,

    -- Average end-to-end delivery time per month
    ROUND(
        AVG(julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp))
    , 2)                                                                     AS avg_delivery_days

FROM olist_orders_dataset
WHERE
    order_status                    = 'delivered'
    AND order_purchase_timestamp       IS NOT NULL
    AND order_delivered_customer_date  IS NOT NULL
    AND order_estimated_delivery_date  IS NOT NULL
GROUP BY
    order_month
ORDER BY
    order_month;
"""

df_monthly = pd.read_sql(query_monthly_trend, conn)
display(df_monthly)

,order_month,total_orders,delayed_orders,delay_rate_pct,avg_delivery_days
0,2016-09,1,1,100.00,54.81
1,2016-10,265,3,1.13,19.60
2,2016-12,1,0,0.00,4.69
3,2017-01,750,23,3.07,12.65
4,2017-02,1653,53,3.21,13.17
5,2017-03,2546,142,5.58,12.95
6,2017-04,2303,181,7.86,14.92
7,2017-05,3545,128,3.61,11.32
8,2017-06,3135,121,3.86,12.01
9,2017-07,3872,133,3.43,11.59



## Query 4 — Logistics Funnel: Seller Fulfilment vs. Carrier Transit


In [7]:
query_funnel = """
SELECT
    strftime('%Y-%m', order_purchase_timestamp)                              AS order_month,

    -- Segment 1 — Seller fulfilment time: payment approval → handover to carrier
    -- Measures how quickly sellers prepare and dispatch orders
    ROUND(
        AVG(julianday(order_delivered_carrier_date) - julianday(order_approved_at))
    , 2)                                                                     AS avg_seller_fulfillment_days,

    -- Segment 2 — Carrier transit time: carrier pickup → customer delivery
    -- This is the core SPX last-mile metric
    ROUND(
        AVG(julianday(order_delivered_customer_date) - julianday(order_delivered_carrier_date))
    , 2)                                                                     AS avg_carrier_transit_days,

    -- Total lead time: sum of both segments for end-to-end view
    ROUND(
        AVG(julianday(order_delivered_customer_date) - julianday(order_approved_at))
    , 2)                                                                     AS total_lead_time_days

FROM olist_orders_dataset
WHERE
    order_status                    = 'delivered'
    AND order_approved_at              IS NOT NULL
    AND order_delivered_carrier_date   IS NOT NULL
    AND order_delivered_customer_date  IS NOT NULL
GROUP BY
    order_month
ORDER BY
    order_month;
"""

df_funnel = pd.read_sql(query_funnel, conn)
display(df_funnel)

,order_month,avg_seller_fulfillment_days,avg_carrier_transit_days,total_lead_time_days
0,2016-09,53.21,1.61,54.81
1,2016-10,13.00,5.73,18.73
2,2016-12,3.28,1.40,4.68
3,2017-01,2.94,9.20,12.15
4,2017-02,3.24,9.54,12.78
5,2017-03,2.87,9.83,12.70
6,2017-04,3.14,11.24,14.38
7,2017-05,2.57,8.34,10.91
8,2017-06,2.74,8.88,11.62
9,2017-07,2.70,8.48,11.19



## Query 5 — Top 10 Routes by Delay Rate (Hub Performance)


In [8]:
query_routes = """
SELECT
    s.seller_state                                                           AS origin_hub,
    c.customer_state                                                         AS destination_hub,
    COUNT(o.order_id)                                                        AS total_volume,

    -- Average carrier transit days specific to this route
    ROUND(
        AVG(julianday(o.order_delivered_customer_date)
            - julianday(o.order_delivered_carrier_date))
    , 2)                                                                     AS route_avg_transit_days,

    -- Route-level delay rate — identifies which lanes are underperforming SLA
    ROUND(
        SUM(CASE
                WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
                THEN 1 ELSE 0
            END) * 100.0 / COUNT(o.order_id)
    , 2)                                                                     AS route_delay_rate_pct

FROM      olist_orders_dataset       o
JOIN      olist_customers_dataset    c  ON o.customer_id  = c.customer_id
JOIN      olist_order_items_dataset  oi ON o.order_id     = oi.order_id
JOIN      olist_sellers_dataset      s  ON oi.seller_id   = s.seller_id
WHERE
    o.order_status                    = 'delivered'
    AND o.order_delivered_customer_date  IS NOT NULL
    AND o.order_delivered_carrier_date   IS NOT NULL
    AND o.order_estimated_delivery_date  IS NOT NULL
GROUP BY
    s.seller_state,
    c.customer_state
HAVING
    -- Minimum volume threshold to filter out statistically insignificant routes
    COUNT(o.order_id) > 100
ORDER BY
    route_delay_rate_pct DESC
LIMIT 10;
"""

df_routes = pd.read_sql(query_routes, conn)
display(df_routes)

,origin_hub,destination_hub,total_volume,route_avg_transit_days,route_delay_rate_pct
0,MA,SP,130,10.65,26.92
1,SP,AL,273,21.23,25.27
2,SP,MA,549,18.40,22.22
3,SP,SE,237,17.70,18.14
4,SP,PI,362,16.94,17.96
5,SP,CE,1091,17.58,15.40
6,PR,BA,163,17.53,15.34
7,SP,RJ,9403,12.60,14.85
8,SP,BA,2626,16.34,14.78
9,SP,ES,1621,12.33,14.00



---
## Part 2: Data Integrity & Deep-Dive Analysis (Pandas)

While the initial SQL queries provide a fast, macro-level view for operations dashboards, raw production databases usually contain noise (e.g., system errors, missing timestamps for lost parcels). 

Before moving into Root Cause Analysis (RCA), we need to perform strict Exploratory Data Analysis (EDA) and clean the dataset. This ensures our final business recommendations are built on solid ground.

In [9]:
import pandas as pd

# Load the core tables needed for delay analysis
orders    = pd.read_csv('olist_orders_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')
reviews   = pd.read_csv('olist_order_reviews_dataset.csv')
items     = pd.read_csv('olist_order_items_dataset.csv')
sellers   = pd.read_csv('olist_sellers_dataset.csv')

In [10]:
print('Data shape (rows × columns):', orders.shape)
print()
print('first 3 lines：')
display(orders.head(3))
print()
print('Field type：')
print(orders.dtypes)

Data shape (rows × columns): (99441, 8)

first 3 lines：


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00



Field type：
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object


### 2.1 EDA & Data Cleaning
We are calculating SLA compliance based on strict delivery timelines. Therefore, any order missing key logistical timestamps (approved at, carrier handover, delivery date) must be investigated and removed to avoid skewing our metrics. We also need to filter out cancelled or incomplete orders, as they don't have a valid end-to-end delivery cycle.

In [11]:
# Check missing values — identifies which columns need cleaning
missing = orders.isnull().sum()

# Filter to only show columns that actually have missing data
missing = missing[missing > 0]

print('Fields with null values：')
print(missing)
print()

# Calculate missing percentage to assess severity
missing_pct = (missing / len(orders) * 100).round(2)
print('percentage of null values（%）：')
print(missing_pct)

Fields with null values：
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

percentage of null values（%）：
order_approved_at                0.16
order_delivered_carrier_date     1.79
order_delivered_customer_date    2.98
dtype: float64


In [12]:


# Keep only delivered orders — cancelled/shipped orders have no actual delivery date
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()

# .copy() prevents SettingWithCopyWarning by creating an independent DataFrame

print(f'All orders：{len(orders):,} row')
print(f'Orders delivered：{len(orders_delivered):,} row')
print(f'Filtered out：{len(orders) - len(orders_delivered):,} row')

All orders：99,441 row
Orders delivered：96,478 row
Filtered out：2,963 row


In [13]:
# Drop rows where key datetime columns are missing — cannot calculate delay without them
time_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

# Record the number of rows before deletion, compare them after deletion, and know how much data was lost.
rows_before = len(orders_delivered)

orders_delivered = orders_delivered.dropna(subset=time_cols)

rows_after = len(orders_delivered)
print(f'Before deletion：{rows_before:,} row')
print(f'After deletion：{rows_after:,} row')
print(f'Discarded：{rows_before - rows_after:,} row（{(rows_before - rows_after)/rows_before*100:.2f}%）')

Before deletion：96,478 row
After deletion：96,455 row
Discarded：23 row（0.02%）


In [14]:
# Convert string columns to datetime — required for arithmetic operations like subtraction
for col in time_cols:
    orders_delivered[col] = pd.to_datetime(orders_delivered[col])
    
print('Converted field type：')
print(orders_delivered[time_cols].dtypes)

Converted field type：
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [15]:
# Cleaning complete, final confirmation
print('===Cleaning complete===')
print(f'Final data：{orders_delivered.shape[0]:,} row × {orders_delivered.shape[1]} conlumns')
print()
print('Time column types confirmation:')
print(orders_delivered[time_cols].dtypes)
print()
print('Remaining null values:')
print(orders_delivered[time_cols].isnull().sum())

===Cleaning complete===
Final data：96,455 row × 8 conlumns

Time column types confirmation:
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

Remaining null values:
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
dtype: int64


In [16]:
# Calculate three key duration metrics that decompose the full delivery timeline

# End-to-end delivery time from purchase to customer receipt
orders_delivered['total_delivery_days'] = (
    orders_delivered['order_delivered_customer_date'] -
    orders_delivered['order_purchase_timestamp']
).dt.days

In [17]:
# Delay days: positive = late, negative = early delivery (buffer)
orders_delivered['delay_days'] = (
    orders_delivered['order_delivered_customer_date'] -
    orders_delivered['order_estimated_delivery_date']
).dt.days

### 2.2 Data Quality Check: Raw SQL vs. Cleaned Pandas Pipeline
This is a critical checkpoint. We are comparing the high-level metrics generated from the raw SQL tables against our strictly cleaned Pandas dataset. 

**Business Takeaway:** Calculating SLA metrics directly on raw, uncleaned data (SQL: 8.11%) inflates the delay rate compared to the cleaned baseline (Pandas: 6.77%). This discrepancy highlights the importance of data governance — anomalous orders (e.g., lost in transit) artificially hurt operations' KPI scores if not properly filtered.

In [18]:
# 3.Seller fulfilment time: from payment approval to carrier handover
orders_delivered['seller_handling_days'] = (
    orders_delivered['order_delivered_carrier_date'] -
    orders_delivered['order_approved_at']
).dt.days

In [19]:
# Binary flag for delayed orders — used for groupby filtering in RCA
orders_delivered['is_delayed'] = (
    orders_delivered['delay_days'] > 0
).astype(int)
# Convert True/False to 1/0 for easier delay calculation using sum()

In [20]:
# Preview the newly computed columns alongside order identifiers
cols_to_check = [
    'order_id',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'total_delivery_days',
    'delay_days',
    'seller_handling_days',
    'is_delayed'
]

display(orders_delivered[cols_to_check].head(10))

,order_id,order_delivered_customer_date,order_estimated_delivery_date,total_delivery_days,delay_days,seller_handling_days,is_delayed
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-10 21:25:13,2017-10-18,8,-8,2,0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-07 15:27:45,2018-08-13,13,-6,0,0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-17 18:06:29,2018-09-04,9,-18,0,0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-02 00:28:42,2017-12-15,13,-13,3,0
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-16 18:17:02,2018-02-26,2,-10,0,0
5,a4591c265e18cb1dcee52889e2d8acc3,2017-07-26 10:57:55,2017-08-01,16,-6,1,0
7,6514b8ad8028c9f2cc2374ded245783f,2017-05-26 12:55:51,2017-06-07,9,-12,5,0
8,76c6e866289321a7c93b82b54852dc33,2017-02-02 14:08:10,2017-03-06,9,-32,1,0
9,e69bfb5eb88e0ed6a785585b27e16dbf,2017-08-16 17:14:30,2017-08-23,18,-7,12,0
10,e6ce16cb79ec1d90b1da9085a6118aeb,2017-05-29 11:18:31,2017-06-07,12,-9,1,0


In [21]:
# Summary statistics to validate the computed columns make logical sense
print('=== Delay Days Summary Statistics ===')
print(orders_delivered[['total_delivery_days',
                        'delay_days',
                        'seller_handling_days']].describe().round(2))

print()
print('=== Delay Overview ===')
total    = len(orders_delivered)
delayed  = orders_delivered['is_delayed'].sum()
ontime   = total - delayed

print(f'Total Orders:     {total:,}')
print(f'Delayed Orders:   {delayed:,}  ({delayed/total*100:.1f}%)')
print(f'On-time Orders:   {ontime:,}  ({ontime/total*100:.1f}%)')
print()
print(f'Average Delay Days (Delayed Orders Only): '
      f'{orders_delivered[orders_delivered["is_delayed"]==1]["delay_days"].mean():.1f} days')

=== Delay Days Summary Statistics ===
       total_delivery_days  delay_days  seller_handling_days
count             96455.00    96455.00              96455.00
mean                 12.09      -11.87                  2.30
std                   9.55       10.18                  3.55
min                   0.00     -147.00               -172.00
25%                   6.00      -17.00                  0.00
50%                  10.00      -12.00                  1.00
75%                  15.00       -7.00                  3.00
max                 209.00      188.00                125.00

=== Delay Overview ===
Total Orders:     96,455
Delayed Orders:   6,533  (6.8%)
On-time Orders:   89,922  (93.2%)

Average Delay Days (Delayed Orders Only): 10.6 days


In [22]:
# Isolate delayed orders and remove data entry anomalies for clean RCA
delayed_df = orders_delivered[
    (orders_delivered['is_delayed'] == 1) &          
    (orders_delivered['seller_handling_days'] >= 0)  
].copy()

print(f'Number of delayed orders used for RCA：{len(delayed_df):,} row')

Number of delayed orders used for RCA：6,509 row


In [23]:
# ── Cross-validate Pandas results against SQL output ──────────────────
# Ensures both pipelines produce consistent numbers — a data integrity check

pandas_delay_rate  = (orders_delivered['is_delayed'].sum() / 
                      len(orders_delivered) * 100).round(2)
pandas_avg_days    = orders_delivered['total_delivery_days'].mean().round(2)

print('=== SQL vs Pandas Cross-Validation ===')
print(f'{"Metric":<25} {"SQL":>10} {"Pandas":>10} {"Match":>8}')
print('-' * 55)
print(f'{"Delay rate (%)":<25} {kpi_delay_rate:>10} {pandas_delay_rate:>10} '
      f'{"✅" if abs(kpi_delay_rate - pandas_delay_rate) < 0.1 else "⚠️":>8}')
print(f'{"Avg delivery days":<25} {kpi_avg_days:>10} {pandas_avg_days:>10} '
      f'{"✅" if abs(kpi_avg_days - pandas_avg_days) < 0.1 else "⚠️":>8}')

print()
# Validate that Pandas RCA top states match SQL top 5 ranking
print('=== Top Delay States: SQL vs Pandas RCA ===')
print(f'SQL top 5    : {sql_top5_states}')

=== SQL vs Pandas Cross-Validation ===
Metric                           SQL     Pandas    Match
-------------------------------------------------------
Delay rate (%)                  8.11       6.77       ⚠️
Avg delivery days              12.56      12.09       ⚠️

=== Top Delay States: SQL vs Pandas RCA ===
SQL top 5    : ['AL', 'MA', 'PI', 'CE', 'SE']


---
## Part 3: Root Cause Analysis (RCA) of Delayed Orders

With a clean dataset showing a true delay rate of 6.8%, we need to isolate the primary bottlenecks. We will evaluate three distinct operational hypotheses:
1. **Seller Operations:** Are delays driven by sellers taking too long to pack and hand over parcels?
2. **Seasonality/Capacity:** Do delay rates spike during specific promotional months due to carrier capacity limits?
3. **Geographical Constraints:** Are specific destination states structurally prone to late deliveries?


In [24]:
# Isolate delayed orders and remove data entry anomalies for clean RCA
delayed_df = orders_delivered[
    (orders_delivered['is_delayed'] == 1) &          # Keep only delayed orders
    (orders_delivered['seller_handling_days'] >= 0)  # Filter out anomalies with negative handling days
].copy()

print(f'Delayed orders used for RCA: {len(delayed_df):,} rows')

Delayed orders used for RCA: 6,509 rows


In [25]:
# RCA Dimension 1: Seller Handling Time
# Hypothesis: Slow seller dispatch -> Late carrier pickup -> Overall delivery delay

# Segment orders by seller handling speed to test if slow fulfilment drives delays
# Cut seller_handling_days into 4 buckets for comparison
delayed_df['handling_segment'] = pd.cut(
    delayed_df['seller_handling_days'],
    bins=[-1, 1, 3, 7, float('inf')],         # Bin edges: 0-1 days / 2-3 days / 4-7 days / 7+ days
    labels=['0-1 days', '2-3 days', '4-7 days', '7+ days'] # Corresponding labels
)

# pd.cut() bins continuous numerical values into discrete intervals; float('inf') represents no upper limit

handling_rca = delayed_df.groupby('handling_segment', observed=True).agg(
    total_orders=('order_id', 'count'),                       # Number of delayed orders in each segment
    avg_delay_days=('delay_days', 'mean'),                    # Average severity of delay for this segment
    avg_handling_days=('seller_handling_days', 'mean')        # Actual average handling days for this segment
).round(2)

print('=== RCA Dimension 1: Seller Handling Time vs Delay ===')
display(handling_rca)

=== RCA Dimension 1: Seller Handling Time vs Delay ===


,total_orders,avg_delay_days,avg_handling_days
handling_segment,,,
0-1 days,2352,10.83,0.54
2-3 days,1529,10.77,2.46
4-7 days,1406,10.23,5.14
7+ days,1222,10.49,17.02


In [26]:

# RCA Dimension 2: Order Peak Periods
# Hypothesis: Sudden spikes in order volume during certain months -> Insufficient carrier capacity -> Increased delay rates

# Extract order month to test whether peak periods correlate with higher delay rates
orders_delivered['order_month'] = orders_delivered[
    'order_purchase_timestamp'
].dt.to_period('M')
# dt.to_period('M') converts datetime to a year-month period format, which is better suited for time-series grouping than strftime

# Monthly statistics: Total volume, delayed volume, and delay rate
monthly_rca = orders_delivered.groupby('order_month').agg(
    total_orders=('order_id', 'count'),
    delayed_orders=('is_delayed', 'sum')             # is_delayed is 1/0, so sum() yields the total number of delayed orders
).reset_index()

# reset_index() converts order_month from an index back to a standard column for easier subsequent operations

monthly_rca['monthly_delay_rate'] = (
    monthly_rca['delayed_orders'] / monthly_rca['total_orders'] * 100
).round(2)

print('=== RCA Dimension 2: Monthly Order Volume vs Delay Rate ===')
display(monthly_rca.sort_values('monthly_delay_rate', ascending=False).head(8))
# Display only the top 8 months with the highest delay rates to focus on problem periods

=== RCA Dimension 2: Monthly Order Volume vs Delay Rate ===


,order_month,total_orders,delayed_orders,monthly_delay_rate
0,2016-09,1,1,100.00
17,2018-03,7003,1328,18.96
16,2018-02,6555,926,14.13
13,2017-11,7288,904,12.40
14,2017-12,5513,411,7.46
6,2017-04,2303,151,6.56
19,2018-05,6749,443,6.56
22,2018-08,6351,393,6.19


In [27]:
# ══════════════════════════════════════════════════════════
# RCA Dimension 3: Destination State (Delivery Distance)
# Hypothesis: Remote states have poorer logistics infrastructure -> longer transit times -> more prone to delays
# ══════════════════════════════════════════════════════════

# Merge state information to test whether geography drives delay rates
orders_with_state = orders_delivered.merge(
    customers[['customer_id', 'customer_state']],  # Only select the two required columns to avoid pulling the whole table
    on='customer_id',
    how='left'   # left join: keep all orders to prevent data loss even if there's no match in the customers table
)

state_rca = orders_with_state.groupby('customer_state').agg(
    total_orders=('order_id', 'count'),
    delayed_orders=('is_delayed', 'sum'),
    avg_total_delivery_days=('total_delivery_days', 'mean'),
    avg_delay_days=('delay_days', 'mean')
).reset_index()

state_rca['state_delay_rate'] = (
    state_rca['delayed_orders'] / state_rca['total_orders'] * 100
).round(2)

state_rca['avg_total_delivery_days'] = state_rca['avg_total_delivery_days'].round(1)
state_rca['avg_delay_days']   = state_rca['avg_delay_days'].round(1)

# Filter out low-volume states to ensure statistical significance
state_rca = state_rca[state_rca['total_orders'] > 100]

print('=== RCA Dimension 3: Destination State vs Delay Rate (TOP 10) ===')
display(state_rca.sort_values('state_delay_rate', ascending=False).head(10))

=== RCA Dimension 3: Destination State vs Delay Rate (TOP 10) ===


,customer_state,total_orders,delayed_orders,avg_total_delivery_days,avg_delay_days,state_delay_rate
1,AL,397,85,24.0,-8.7,21.41
9,MA,716,125,21.1,-9.5,17.46
24,SE,335,51,21.0,-10.0,15.22
16,PI,476,66,19.0,-11.3,13.87
5,CE,1278,176,20.8,-10.8,13.77
4,BA,3256,396,18.9,-10.8,12.16
18,RJ,12348,1495,14.8,-11.8,12.11
13,PA,946,106,23.3,-14.1,11.21
7,ES,1995,214,15.3,-10.5,10.73
14,PB,517,54,20.0,-13.3,10.44


In [28]:
# RCA Summary: Which factor has the biggest impact?

# Summarise findings across all three RCA dimensions for reporting
print('=' * 50)
print('RCA Summary Conclusions')
print('=' * 50)

# Impact of seller handling time: Average delay days for orders taking 7+ days to process
slow_seller_delay = delayed_df[
    delayed_df['seller_handling_days'] > 7
]['delay_days'].mean()

fast_seller_delay = delayed_df[
    delayed_df['seller_handling_days'] <= 1
]['delay_days'].mean()

print(f'\n[Seller Handling Time]')
print(f'  Delayed orders with ≤ 1 day handling time, average delay: {fast_seller_delay:.1f} days')
print(f'  Delayed orders with > 7 days handling time, average delay: {slow_seller_delay:.1f} days')

# Delay rate gap between peak months vs. normal months
peak_rate   = monthly_rca.nlargest(3, 'monthly_delay_rate')['monthly_delay_rate'].mean()
normal_rate = monthly_rca.nsmallest(3, 'monthly_delay_rate')['monthly_delay_rate'].mean()

print(f'\n[Order Peak Periods]')
print(f'  Top 3 months with highest delay rate, average: {peak_rate:.1f}%')
print(f'  Bottom 3 months with lowest delay rate, average: {normal_rate:.1f}%')

# Highest vs. lowest delay rate by state
worst_state = state_rca.nlargest(1, 'state_delay_rate').iloc[0]
best_state  = state_rca.nsmallest(1, 'state_delay_rate').iloc[0]

print(f'\n[Destination State]')
print(f'  State with highest delay rate: {worst_state["customer_state"]} ({worst_state["state_delay_rate"]}%)')
print(f'  State with lowest delay rate: {best_state["customer_state"]} ({best_state["state_delay_rate"]}%)')

RCA Summary Conclusions

[Seller Handling Time]
  Delayed orders with ≤ 1 day handling time, average delay: 10.8 days
  Delayed orders with > 7 days handling time, average delay: 10.5 days

[Order Peak Periods]
  Top 3 months with highest delay rate, average: 44.4%
  Bottom 3 months with lowest delay rate, average: 0.6%

[Destination State]
  State with highest delay rate: AL (21.41%)
  State with lowest delay rate: AM (2.76%)


---
## Part 4: Quantifying the Business Impact (Delay vs. CSAT)

Operations metrics (like SLA compliance) only matter if they impact the customer. In this final section, we merge delivery performance data with customer review scores. 

**Objective:** To quantify the exact cost of a late delivery in terms of customer dissatisfaction. By measuring how much a 1-star review rate increases per additional day of delay, we can justify budget requests for increasing last-mile logistics capacity.

In [29]:
# Merge delayed flag and delay_days into reviews data
# reviews table contains the customer satisfaction scores we want to analyse
orders_for_review = orders_delivered[
    ['order_id', 'is_delayed', 'delay_days', 'total_delivery_days']
]

reviews_merged = reviews.merge(
    orders_for_review,
    on='order_id',
    how='inner'  # inner join: only keep orders that exist in both tables
)

print(f'Orders with review data: {len(reviews_merged):,}')
print(f'Score distribution:\n{reviews_merged["review_score"].value_counts().sort_index()}')

Orders with review data: 96,338
Score distribution:
review_score
1     9405
2     2940
3     7960
4    18983
5    57050
Name: count, dtype: int64


In [30]:
# Compare average review score between on-time and delayed orders
# This directly tests whether delay is correlated with customer dissatisfaction
ontime_score  = reviews_merged[reviews_merged['is_delayed'] == 0]['review_score'].mean()
delayed_score = reviews_merged[reviews_merged['is_delayed'] == 1]['review_score'].mean()

print('=== On-time vs Delayed: Average Review Score ===')
print(f'On-time orders  : {ontime_score:.2f} / 5.0')
print(f'Delayed orders  : {delayed_score:.2f} / 5.0')
print(f'Score drop      : {ontime_score - delayed_score:.2f} points')

=== On-time vs Delayed: Average Review Score ===
On-time orders  : 4.29 / 5.0
Delayed orders  : 2.27 / 5.0
Score drop      : 2.02 points


In [31]:
# Calculate 1-star review rate for on-time vs delayed orders
# 1-star rate is the most actionable metric — it represents customers who are actively unhappy
ontime_1star = (
    reviews_merged[reviews_merged['is_delayed'] == 0]['review_score'] == 1
).mean() * 100

delayed_1star = (
    reviews_merged[reviews_merged['is_delayed'] == 1]['review_score'] == 1
).mean() * 100

print('=== 1-Star Review Rate ===')
print(f'On-time orders : {ontime_1star:.1f}%')
print(f'Delayed orders : {delayed_1star:.1f}%')
print(f'Multiplier     : {delayed_1star / ontime_1star:.1f}x more likely to get 1-star when delayed')

=== 1-Star Review Rate ===
On-time orders : 6.6%
Delayed orders : 53.7%
Multiplier     : 8.1x more likely to get 1-star when delayed


In [32]:
# Segment delayed orders by severity to see if longer delays cause worse reviews
# Tests whether the relationship is linear: more delay = worse score
reviews_merged['delay_segment'] = pd.cut(
    reviews_merged['delay_days'],
    bins=[-float('inf'), 0, 3, 7, 14, float('inf')],
    labels=['On-time', '1-3 days late', '4-7 days late', '8-14 days late', '14+ days late']
)

delay_vs_score = reviews_merged.groupby('delay_segment', observed=True).agg(
    order_count    = ('order_id', 'count'),
    avg_score      = ('review_score', 'mean'),
    pct_1star      = ('review_score', lambda x: (x == 1).mean() * 100)
).round(2)

print('=== Delay Severity vs Review Score ===')
display(delay_vs_score)

=== Delay Severity vs Review Score ===


,order_count,avg_score,pct_1star
delay_segment,,,
On-time,89930,4.29,6.63
1-3 days late,1856,3.29,25.16
4-7 days late,1755,2.10,58.58
8-14 days late,1452,1.68,70.39
14+ days late,1345,1.72,68.92


In [33]:
# Final summary: quantify the business impact of delay on customer satisfaction
print('=' * 55)
print('Delay → Review Score Impact Summary')
print('=' * 55)

worst = delay_vs_score['avg_score'].idxmin()  # segment with lowest average score
print(f'\nWorst performing segment : {worst}')
print(f'Average score (on-time)  : {delay_vs_score.loc["On-time", "avg_score"]}')
print(f'Average score ({worst}) : {delay_vs_score.loc[worst, "avg_score"]}')
print(f'1-star rate (on-time)    : {delay_vs_score.loc["On-time", "pct_1star"]}%')
print(f'1-star rate ({worst})    : {delay_vs_score.loc[worst, "pct_1star"]}%')

Delay → Review Score Impact Summary

Worst performing segment : 8-14 days late
Average score (on-time)  : 4.29
Average score (8-14 days late) : 1.68
1-star rate (on-time)    : 6.63%
1-star rate (8-14 days late)    : 70.39%


In [34]:
# ── Export Comprehensive Dataset for Tableau ───────────────────────
# We need to bring 'customer_state' and 'order_month' back into our dataset
# by merging them from the previously created 'orders_with_state' dataframe.

# 1. Merge the required geographic and time dimensions into the review data
tableau_base = reviews_merged.merge(
    orders_with_state[['order_id', 'customer_state', 'order_month']],
    on='order_id',
    how='left'
)

# 2. Select the final columns needed for the Tableau dashboard
tableau_export = tableau_base[[
    'order_id', 
    'is_delayed', 
    'delay_days', 
    'delay_segment',
    'review_score', 
    'total_delivery_days', 
    'customer_state', 
    'order_month'
]].copy() # Use .copy() to avoid Pandas SettingWithCopyWarning

# 3. Convert 'order_month' (Pandas Period type) to string for Tableau compatibility
# Tableau parses standard string dates ('2017-11') much better than Pandas period objects
tableau_export['order_month'] = tableau_export['order_month'].astype(str)

# 4. Export to CSV (using professional snake_case naming)
export_filename = 'logistics_sla_and_csat_metrics.csv'
tableau_export.to_csv(export_filename, index=False)

print(f" Tableau export complete: {export_filename}")
print(f"Data shape: {tableau_export.shape[0]:,} rows x {tableau_export.shape[1]} columns")

 Tableau export complete: logistics_sla_and_csat_metrics.csv
Data shape: 96,338 rows x 8 columns
